# Adaptive Compute SSM — Phase 2: Retrieval-Based MoE Processing Layer
## Heterogeneous Expert Pool · KNN Routing · Distance-Weighted Mixing · Delta Formulation

**Phase 2 changes vs Phase 1:**
- `ProcessingLayer` (monolithic MLP) → `MoEProcessingLayer`
- **3 heterogeneous expert types**: GELU-MLP, SiLU-MLP, Polynomial — cycled across the expert pool
- **KNN routing**: router head produces a query in latent space; top-K experts selected by L2 distance to learned centroids
- **Distance-based RBF mixing weights**: `w_i = exp(-d_i² / T²)` normalised over top-K — 100 % at d=0, 0 % at d→∞, fully differentiable
- **Delta formulation**: each expert predicts a residual `Δ`; output is `z + Σ wᵢ·Δᵢ` — gradient highway through the identity path
- **Centroid repulsion loss**: keeps expert centroids spread in key space

Everything else (InputLayer, GateMLP, OutputLayer, TBPTT, KS anchoring, log-norm regularisation) is identical to Phase 1.


In [ ]:
# Install all dependencies first
# comet_ml MUST be installed and imported BEFORE PyTorch
!pip install -q comet_ml datasets transformers tokenizers tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.2/786.2 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.0 MB/s eta 0:00:00


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  IMPORTANT: comet_ml MUST be imported BEFORE torch / transformers       ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import comet_ml
from comet_ml import Experiment
from comet_ml.integration.pytorch import log_model

from google.colab import userdata
COMET_KEY = userdata.get('COMET_KEY')

experiment = Experiment(
    api_key      = COMET_KEY,
    project_name = 'adaptive-ssm',
    workspace='irsotarriva'
)

import math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm
from transformers import AutoTokenizer
from datasets import load_dataset
from huggingface_hub import login

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/irsotarriva/adaptive-ssm/6ebe888e2ff949b99d17364cb7b6123f



Device: cuda
GPU: Tesla T4


In [ ]:
from google.colab import userdata
HF_KEY = userdata.get('HF_KEY')
login(token=HF_KEY)


---
## Architecture Overview — Phase 2 MoE

```
token_id ─► Embedding ─► e_t ─────────────────────────────────────────────────────────────────────────┐
                                                                                                        │
h_{t-1} ──────────────────────────────────────────────────────────────────────────────────────────────►│
                                                                                                        ▼
                                                                                              InputLayer (once)
                                                                                                        │ z_0
                                                                                                        ▼
                                                              ┌──────────────────────────── GateMLP ◄── z_k
                                                              │                        s_k ~ N(0,1)
                                                              │  Φ(s_k) < threshold?
                                                              │  YES → OutputLayer
                                                              │  NO  ↓
                                                              │   MoEProcessingLayer (shared weights, called recurrently)
                                                              │     ├─ RouterHead: z_k → query q
                                                              │     ├─ KNN: top-K experts by ||q - centroid_i||₂
                                                              │     ├─ RBF weights: wᵢ = exp(-dᵢ²/T²) / Σ exp(-dⱼ²/T²)
                                                              │     ├─ Expert deltas: Δᵢ = expert_i(z_k)  [heterogeneous types]
                                                              │     └─ Output: z_{k+1} = z_k + Σ wᵢ·Δᵢ
                                                              └──────────────────────────── z_{k+1} ──►(loop)
                                                                                                        │ z_K
                                                                                                        ▼
                                                                                           OutputLayer (once)
                                                                                              │         │
                                                                                            logits    h_t
```

**Key invariant preserved from Phase 1:** SSM state `h` is read once (InputLayer) and written once (OutputLayer) per token.
The MoE processing layer — like the Phase 1 MLP — operates entirely in the internal scratch space.


---
## 1. Heterogeneous Expert Types

All experts share the same interface: `forward(z) → Δ` where `Δ` is a **delta** (residual)
in the same space as `z`. The residual connection lives at the MoE layer level:

```
z_{k+1} = z_k + Σ_i  w_i · expert_i(z_k)
```

This formulation guarantees gradient flow: ∂z_{k+1}/∂z_k = I + Σ wᵢ · J_i — the identity
Jacobian is always present, preventing vanishing gradients no matter how small the expert
contributions are.

### Expert type catalogue (Phase 2)
| Type | Internal structure | Inductive bias |
|------|--------------------|----------------|
| `MLPExpert(gelu)` | Linear → GELU → Linear | Smooth nonlinearity, good default |
| `MLPExpert(silu)` | Linear → SiLU → Linear | Gate-like activation, captures sparse features |
| `PolynomialExpert` | [z, z²] → Linear | Quadratic interactions, no depth needed |

The pool cycles through types: experts 0,3 are GELU, experts 1,4 are SiLU, experts 2,5 are Polynomial.


In [ ]:
class MLPExpert(nn.Module):
    """
    Small MLP expert with configurable activation.
    Predicts a DELTA (residual) — does NOT include a skip connection.
    The skip connection is applied at the MoEProcessingLayer level.
    """
    def __init__(self, d_internal: int, d_hidden: int, activation: str = 'gelu'):
        super().__init__()
        act = nn.GELU() if activation == 'gelu' else nn.SiLU()
        self.norm = nn.LayerNorm(d_internal)
        self.net  = nn.Sequential(
            nn.Linear(d_internal, d_hidden, bias=False),
            act,
            nn.Linear(d_hidden, d_internal, bias=False),
        )
        # Zero-init output projection: experts start as identity (zero delta)
        nn.init.zeros_(self.net[-1].weight)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        """z: (batch, d_internal) → delta: (batch, d_internal)"""
        return self.net(self.norm(z))


class PolynomialExpert(nn.Module):
    """
    Quadratic feature expansion followed by a linear projection.
    Computes [z_norm, z_norm²] → Linear → delta.

    No hidden layer or nonlinearity beyond the quadratic expansion.
    Captures multiplicative feature interactions that MLPs need depth to learn.
    """
    def __init__(self, d_internal: int):
        super().__init__()
        self.norm = nn.LayerNorm(d_internal)
        # Input: z concatenated with z^2 → 2*d_internal features
        self.proj = nn.Linear(d_internal * 2, d_internal, bias=False)
        nn.init.zeros_(self.proj.weight)  # zero-init: start as identity

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        """z: (batch, d_internal) → delta: (batch, d_internal)"""
        z_n = self.norm(z)
        return self.proj(torch.cat([z_n, z_n.pow(2)], dim=-1))


---
## 2. Retrieval-Based MoE Router

### Routing mechanism

1. **Query projection**: `q = W_q · z_k` — projects current representation into routing space
2. **KNN selection**: compute L2 distances `d_i = ||q - centroid_i||₂` for all experts; select top-K nearest
3. **RBF mixing weights** (differentiable):

$$w_i = \frac{\exp(-d_i^2 / T^2)}{\sum_{j \in \text{top-K}} \exp(-d_j^2 / T^2)}$$

**Properties:**
- `w_i → 1` as `d_i → 0` (centroid exactly matches query)
- `w_i → 0` as `d_i → ∞` (far expert gets no weight)
- Weights sum to 1 within the top-K selection
- Fully differentiable w.r.t. both query `q` and centroids `cᵢ` — gradients flow into both

### Hard vs. soft selection
Top-K selection is a hard (non-differentiable) discrete step.
The **weights** for the selected experts are differentiable. This is the same trade-off used in
all practical sparse MoE models (e.g., GShard, Switch Transformer) and works well in practice.

### Centroid repulsion
Expert centroids are regularised by a pairwise repulsion loss:

$$L_{\text{rep}} = \frac{1}{n(n-1)} \sum_{i \ne j} \max(0,\, m - d_{ij})^2$$

This penalises centroids that are closer than margin `m`, keeping the key space spread out.


In [ ]:
class MoERouter(nn.Module):
    """
    KNN-based differentiable router.

    Each expert has a learned centroid in the same space as the query projection.
    Top-K experts are selected by L2 distance; mixing weights use an RBF kernel
    normalised over the selected set.

    Gradient flow:
      - Weights are differentiable w.r.t. distances, which are differentiable
        w.r.t. both the query (W_q · z) and the centroids.
      - The hard top-K selection is not differentiable, but the centroid
        repulsion loss provides a direct gradient signal on centroids.
    """

    def __init__(
        self,
        d_internal  : int,
        n_experts   : int,
        top_k       : int,
        temperature : float = 1.0,
    ):
        super().__init__()
        self.n_experts   = n_experts
        self.top_k       = min(top_k, n_experts)
        self.temperature = temperature

        # Query projection: maps z_k into routing space
        self.query_proj = nn.Linear(d_internal, d_internal, bias=False)

        # Expert centroids: (n_experts, d_internal)
        # Initialise spread on a unit sphere to encourage good coverage from the start
        centroids = torch.randn(n_experts, d_internal)
        centroids = F.normalize(centroids, dim=-1)
        self.centroids = nn.Parameter(centroids)

    def forward(self, z: torch.Tensor):
        """
        z: (batch, d_internal)

        Returns:
            weights   : (batch, top_k)  — RBF-normalised mixing weights, sum=1
            topk_idx  : (batch, top_k)  — expert indices (int64)
            distances : (batch, n_experts) — full distance matrix (for logging)
        """
        q = self.query_proj(z)  # (batch, d_internal)

        # Pairwise L2 distances: (batch, n_experts)
        diff      = q.unsqueeze(1) - self.centroids.unsqueeze(0)  # (B, E, D)
        distances = diff.norm(dim=-1)                              # (B, E)

        # Hard top-K selection (smallest distances = nearest centroids)
        topk_dist, topk_idx = distances.topk(self.top_k, dim=-1, largest=False)
        # topk_dist: (batch, top_k),  topk_idx: (batch, top_k)

        # RBF weights — differentiable w.r.t. topk_dist
        # w_i = exp(-d_i^2 / T^2),  then normalise over top-K
        rbf     = torch.exp(-topk_dist.pow(2) / (self.temperature ** 2 + 1e-8))
        weights = rbf / (rbf.sum(dim=-1, keepdim=True) + 1e-8)   # (batch, top_k)

        return weights, topk_idx, distances


---
## 3. MoE Processing Layer

The `MoEProcessingLayer` replaces the single `ProcessingLayer` from Phase 1.
Its external interface is identical (same in/out shapes), so `AdaptiveSSM`'s depth loop
requires only minimal changes.

### Why compute all expert deltas?
For a small fixed pool (6 experts on a Colab GPU), computing all expert outputs in a
batched `torch.stack` is more GPU-efficient than conditionally routing subsets.
The top-K gather then selects the relevant deltas without any Python loops over the batch.

This approach will need revisiting if the pool grows large (Phase 4 lifecycle).


In [ ]:
class MoEProcessingLayer(nn.Module):
    """
    Retrieval-based MoE processing layer.

    Expert pool is heterogeneous: types cycle GELU-MLP / SiLU-MLP / Polynomial.
    All experts predict DELTAS; the skip connection is applied here:

        z_{k+1} = z_k + Σ_i  w_i · expert_i(z_k)

    This object is instantiated ONCE and called recurrently at every depth step.
    All depth applications share the same expert weights and centroids.

    Returns (z_next, routing_info) — routing_info is a dict for logging/debugging.
    """

    # Expert type cycle (repeats for pools larger than 3)
    _TYPE_CYCLE = ['gelu', 'silu', 'poly']

    def __init__(
        self,
        d_internal  : int,
        d_expert    : int,   # hidden dim inside MLP experts
        n_experts   : int,
        top_k       : int,
        temperature : float = 1.0,
    ):
        super().__init__()
        self.n_experts = n_experts
        self.top_k     = top_k

        # Build heterogeneous expert pool
        experts = []
        for i in range(n_experts):
            t = self._TYPE_CYCLE[i % len(self._TYPE_CYCLE)]
            if t == 'poly':
                experts.append(PolynomialExpert(d_internal))
            else:
                experts.append(MLPExpert(d_internal, d_expert, activation=t))
        self.experts = nn.ModuleList(experts)

        # Labels for logging (human-readable expert types)
        self.expert_labels = [
            self._TYPE_CYCLE[i % len(self._TYPE_CYCLE)] for i in range(n_experts)
        ]

        self.router = MoERouter(d_internal, n_experts, top_k, temperature)

    def forward(self, z: torch.Tensor):
        """
        z: (batch, d_internal)

        Returns:
            z_next       : (batch, d_internal)
            routing_info : dict  — detached tensors for logging
        """
        weights, topk_idx, distances = self.router(z)
        # weights:   (batch, top_k)    — differentiable mixing weights
        # topk_idx:  (batch, top_k)    — int64 expert indices
        # distances: (batch, n_experts)

        # ── Compute ALL expert deltas in one batched pass ──────────────────
        # all_deltas: (batch, n_experts, d_internal)
        all_deltas = torch.stack([exp(z) for exp in self.experts], dim=1)

        # ── Gather top-K selected deltas ───────────────────────────────────
        # idx_expanded: (batch, top_k, d_internal)
        idx_expanded    = topk_idx.unsqueeze(-1).expand(-1, -1, z.shape[-1])
        selected_deltas = all_deltas.gather(1, idx_expanded)   # (batch, top_k, d_internal)

        # ── Weighted sum of deltas (differentiable w.r.t. weights + deltas) ─
        weighted_delta = (weights.unsqueeze(-1) * selected_deltas).sum(dim=1)
        # weighted_delta: (batch, d_internal)

        # ── Residual: delta projection over initial condition ─────────────
        z_next = z + weighted_delta

        routing_info = {
            'weights'   : weights.detach(),    # (batch, top_k)
            'topk_idx'  : topk_idx.detach(),   # (batch, top_k)
            'distances' : distances.detach(),  # (batch, n_experts)
        }

        return z_next, routing_info


---
## 4. Input / Gate / Output Layers  *(unchanged from Phase 1)*

These three layers are architecturally identical to Phase 1.
The only change is that `forward_token` now unpacks the extra `routing_info` return value
from `MoEProcessingLayer`.


In [ ]:
class InputLayer(nn.Module):
    """
    Reads SSM state h_{t-1} and token embedding e_t.
    Projects them into the internal processing space z_0.
    Executed exactly once per token.
    """
    def __init__(self, d_embed: int, d_state: int, d_internal: int):
        super().__init__()
        self.proj = nn.Linear(d_embed + d_state, d_internal, bias=False)
        self.norm = nn.LayerNorm(d_internal)

    def forward(self, e_t: torch.Tensor, h: torch.Tensor) -> torch.Tensor:
        return self.norm(self.proj(torch.cat([e_t, h], dim=-1)))


class GateMLP(nn.Module):
    """
    Depth gate: evaluates whether z_k requires further processing.
    Produces a scalar score s ~ N(0,1) (enforced by KS anchoring loss).
    P(go deeper) = Φ(s)  —  standard normal CDF.

    Inference Smirnov shift μ: P(go deeper) = Φ(s + μ)
      μ > 0 → deeper processing   μ < 0 → faster exit   μ = 0 → training working point
    """
    def __init__(self, d_internal: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(d_internal),
            nn.Linear(d_internal, d_internal // 2, bias=False),
            nn.GELU(),
            nn.Linear(d_internal // 2, 1, bias=False)
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z).squeeze(-1)

    @staticmethod
    def normal_cdf(s: torch.Tensor) -> torch.Tensor:
        return 0.5 * (1.0 + torch.erf(s / math.sqrt(2.0)))


class OutputLayer(nn.Module):
    """
    Projects internal representation to vocabulary logits and updates the SSM state.
    Executed exactly once per token at the final processing depth z_K.
    """
    def __init__(self, d_internal: int, d_state: int, vocab_size: int):
        super().__init__()
        self.norm         = nn.LayerNorm(d_internal)
        self.logit_proj   = nn.Linear(d_internal, vocab_size, bias=False)
        self.W_gate       = nn.Linear(d_internal + d_state, d_state, bias=False)
        self.W_candidate  = nn.Linear(d_internal, d_state, bias=False)

    def get_logits(self, z: torch.Tensor) -> torch.Tensor:
        return self.logit_proj(self.norm(z))

    def update_state(self, z: torch.Tensor, h: torch.Tensor) -> torch.Tensor:
        gate      = torch.sigmoid(self.W_gate(torch.cat([z, h], dim=-1)))
        candidate = torch.tanh(self.W_candidate(z))
        return gate * h + (1.0 - gate) * candidate


---
## 5. Full Model — AdaptiveSSMMoE

The main change from Phase 1 is:
- `self.processing_layer = MoEProcessingLayer(...)` instead of `ProcessingLayer`
- `forward_token` returns a 4-tuple `(all_logits, gate_scores, h_new, routing_infos)`
- `generate` ignores routing_info (uses `_` unpacking)


In [ ]:
class AdaptiveSSMMoE(nn.Module):
    """
    Adaptive Compute SSM — Phase 2 (MoE processing layer).

    Architecture (strictly separated roles):
      InputLayer         : (e_t, h_{t-1}) → z_0              [once per token]
      GateMLP            : z_k → scalar s_k                   [at each depth]
      MoEProcessingLayer : z_k → z_{k+1}  (shared weights)   [0..D times]
      OutputLayer        : z_K → logits + h_t                  [once per token]

    SSM state is the ONLY inter-token memory.
    Processing layer has NO access to h.
    """

    def __init__(
        self,
        vocab_size  : int,
        d_embed     : int,
        d_state     : int,
        d_internal  : int,
        d_expert    : int,
        n_experts   : int,
        top_k       : int,
        max_depth   : int = 1,
        temperature : float = 1.0,
    ):
        super().__init__()
        self.d_state   = d_state
        self.max_depth = max_depth

        self.embedding         = nn.Embedding(vocab_size, d_embed)
        self.input_layer       = InputLayer(d_embed, d_state, d_internal)
        self.gate              = GateMLP(d_internal)
        self.processing_layer  = MoEProcessingLayer(
            d_internal, d_expert, n_experts, top_k, temperature
        )
        self.output_layer      = OutputLayer(d_internal, d_state, vocab_size)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.5)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)
        # Re-apply zero-init on expert output projections (done in expert __init__,
        # but xavier_uniform above would overwrite them — re-zero here)
        for exp in self.processing_layer.experts:
            if isinstance(exp, MLPExpert):
                nn.init.zeros_(exp.net[-1].weight)
            elif isinstance(exp, PolynomialExpert):
                nn.init.zeros_(exp.proj.weight)

    def init_state(self, batch_size: int, device) -> torch.Tensor:
        return torch.zeros(batch_size, self.d_state, device=device)

    # ── Training: full-unroll forward for one token position ──────────────
    def forward_token(self, token_ids, h, max_depth):
        """
        Full-unroll forward pass — always processes to max_depth (training mode).

        Returns:
            all_logits    : list[(batch, vocab_size)], length = max_depth + 1
            gate_scores   : list[(batch,)],            length = max_depth
            h_new         : (batch, d_state)
            routing_infos : list[dict],                length = max_depth
        """
        e_t = self.embedding(token_ids)
        z   = self.input_layer(e_t, h)

        all_logits    = []
        gate_scores   = []
        routing_infos = []

        for _ in range(max_depth):
            s_k = self.gate(z)
            gate_scores.append(s_k)
            all_logits.append(self.output_layer.get_logits(z))
            z, rinfo = self.processing_layer(z)
            routing_infos.append(rinfo)

        # Forced exit logits at final depth
        all_logits.append(self.output_layer.get_logits(z))
        h_new = self.output_layer.update_state(z, h)

        return all_logits, gate_scores, h_new, routing_infos

    # ── Inference: early-exit with Smirnov depth control ──────────────────
    @torch.no_grad()
    def generate(
        self,
        prompt_ids     : torch.Tensor,
        max_new_tokens : int   = 100,
        mu             : float = 0.0,
        temperature    : float = 1.0,
        top_k          : int   = 50,
    ) -> torch.Tensor:
        """
        Autoregressive generation with Smirnov shift μ for depth control.
        μ > 0 → deeper thinking,  μ < 0 → faster exit,  μ = 0 → training default.
        """
        self.eval()
        dev = next(self.parameters()).device
        h   = self.init_state(1, dev)
        generated = prompt_ids.to(dev)

        # Warm up SSM state on prompt tokens
        for t in range(generated.shape[1] - 1):
            _, _, h, _ = self.forward_token(generated[:, t], h, self.max_depth)

        for _ in range(max_new_tokens):
            e_t = self.embedding(generated[:, -1])
            z   = self.input_layer(e_t, h)

            # Early-exit depth loop (inference)
            for _ in range(self.max_depth):
                s = self.gate(z)
                if torch.rand(1, device=dev).item() > GateMLP.normal_cdf(s + mu).mean().item():
                    break
                z, _ = self.processing_layer(z)

            logits = self.output_layer.get_logits(z) / temperature
            h      = self.output_layer.update_state(z, h)

            if top_k > 0:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, -1:]] = float('-inf')

            next_tok  = torch.multinomial(F.softmax(logits, dim=-1), 1)
            generated = torch.cat([generated, next_tok], dim=1)

        return generated


---
## 6. Training Objective

$$L = L_{\text{LM}} + \lambda_{\text{KS}} \cdot D(p_s \| \mathcal{N}(0,1)) + \lambda_{\text{norm}} \cdot (\ln \|h_t\|)^2 + \lambda_{\text{rep}} \cdot L_{\text{repulsion}}$$

| Term | Role |
|------|------|
| `L_LM` | Weighted cross-entropy over all exit depths (unchanged from Phase 1) |
| `D(p_s ‖ N(0,1))` | KS anchoring: keeps gate score distribution at N(0,1) |
| `(ln ‖h_t‖)²` | Soft log-norm: prevents SSM state norm explosion |
| `L_repulsion` | **NEW**: pushes expert centroids apart in key space |

### Centroid repulsion loss

$$L_{\text{rep}} = \frac{1}{n(n-1)} \sum_{i \ne j} \max(0,\; m - d_{ij})^2$$

where $d_{ij} = \|c_i - c_j\|_2$ is the centroid pair distance and $m$ is the margin.
This penalises centroids that are closer than the margin, keeping the routing key space spread out.
Unlike load-balancing losses (which depend on runtime statistics), this acts directly on parameters
and is stable from the first step.


In [ ]:
def compute_lm_loss(all_logits, gate_scores, targets, max_depth):
    """
    Full-unroll weighted LM loss.
    L_LM = Σ_k  P(exit at k) · CE(logits_k, target)
    Fully differentiable — exit probabilities are products of Φ(s_k).
    """
    device = targets.device
    batch  = targets.shape[0]
    p_continue = [GateMLP.normal_cdf(s) for s in gate_scores]

    p_reach    = torch.ones(batch, device=device)
    total_loss = torch.zeros(1, device=device)

    for k in range(max_depth):
        p_exit     = p_reach * (1.0 - p_continue[k])
        ce_k       = F.cross_entropy(all_logits[k], targets, reduction='none')
        total_loss = total_loss + (p_exit * ce_k).mean()
        p_reach    = p_reach * p_continue[k]

    ce_final   = F.cross_entropy(all_logits[max_depth], targets, reduction='none')
    total_loss = total_loss + (p_reach * ce_final).mean()
    return total_loss


def ks_anchoring_loss(gate_scores_list):
    """
    KS-style anchoring: moment penalties + CvM shape loss.
    Returns (mean_loss, std_loss) separately for per-term logging.
    """
    scores     = torch.cat([s.flatten() for s in gate_scores_list])
    mean_loss  = scores.mean() ** 2
    std_loss   = (scores.std() - 1.0) ** 2
    sorted_s   = torch.sort(scores).values
    n          = scores.shape[0]
    emp_cdf    = torch.arange(1, n + 1, dtype=scores.dtype, device=scores.device) / n
    tgt_cdf    = 0.5 * (1.0 + torch.erf(sorted_s / math.sqrt(2.0)))
    shape_loss = torch.mean((emp_cdf - tgt_cdf) ** 2)
    return mean_loss + shape_loss, std_loss


def norm_regularization_loss(h):
    """
    Soft log-norm regularisation: L = mean( ln(‖h_t‖) )²
    Minimum at unit norm. Symmetric in log space. Gradient ∝ ln‖h‖/‖h‖.
    """
    norms = torch.norm(h, dim=-1)
    return torch.mean(torch.log(norms + 1e-8) ** 2)


def centroid_repulsion_loss(centroids, margin=2.0):
    """
    Pairwise repulsion on expert centroids.
    L_rep = mean_{i≠j}  max(0, margin - ||c_i - c_j||)²

    Penalises centroids closer than `margin`, pushing them apart.
    Acts directly on parameters — more stable than load-balancing losses.
    """
    n    = centroids.shape[0]
    diff = centroids.unsqueeze(0) - centroids.unsqueeze(1)   # (n, n, d)
    dist = diff.norm(dim=-1)                                  # (n, n)
    mask = (1.0 - torch.eye(n, device=centroids.device))     # exclude self-pairs
    rep  = (torch.clamp(margin - dist, min=0.0) ** 2) * mask
    return rep.sum() / (n * (n - 1))


---
## 7. Dataset — TinyStories  *(unchanged from Phase 1)*


In [ ]:
class TinyStoriesDataset(Dataset):
    """
    Streams TinyStories from HuggingFace.
    Returns (input_ids, target_ids) pairs of fixed length seq_len.
    """
    def __init__(self, split: str, seq_len: int, max_stories=None):
        self.seq_len = seq_len
        tokenizer    = AutoTokenizer.from_pretrained('gpt2')
        tokenizer.pad_token = tokenizer.eos_token
        ds           = load_dataset('roneneldan/TinyStories', split=split, streaming=False)
        if max_stories:
            ds = ds.select(range(min(max_stories, len(ds))))
        self.samples = []
        for row in tqdm(ds, desc=f'Tokenising {split}', leave=False):
            ids = tokenizer.encode(row['text'], truncation=True, max_length=seq_len + 1)
            if len(ids) < 2:
                continue
            if len(ids) < seq_len + 1:
                ids = ids + [tokenizer.eos_token_id] * (seq_len + 1 - len(ids))
            ids = ids[:seq_len + 1]
            self.samples.append(torch.tensor(ids, dtype=torch.long))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ids = self.samples[idx]
        return ids[:-1], ids[1:]   # (input_ids, target_ids), each length seq_len


---
## 8. Training Loop — TBPTT

Identical structure to Phase 1, with these additions:
- **Centroid repulsion loss** added to the total loss
- **Expert utilisation logging**: tracks how often each expert is selected per chunk
- **Routing diagnostics**: mean routing distance, weight entropy logged to Comet ML


In [ ]:
def train(config: dict, experiment=None) -> nn.Module:
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    if experiment:
        experiment.log_parameters(config)

    tokenizer = AutoTokenizer.from_pretrained(config['tokenizer'])
    tokenizer.pad_token = tokenizer.eos_token
    vocab_size = tokenizer.vocab_size

    model = AdaptiveSSMMoE(
        vocab_size  = vocab_size,
        d_embed     = config['d_embed'],
        d_state     = config['d_state'],
        d_internal  = config['d_internal'],
        d_expert    = config['d_expert'],
        n_experts   = config['n_experts'],
        top_k       = config['top_k'],
        max_depth   = config['max_depth'],
        temperature = config['router_temperature'],
    ).to(dev)

    n_params = sum(p.numel() for p in model.parameters())
    print(f'Model parameters: {n_params:,}')
    n_experts = config['n_experts']
    print(f'Expert pool: {n_experts} experts ({model.processing_layer.expert_labels})')
    if experiment:
        experiment.log_parameter('n_params', n_params)

    train_ds = TinyStoriesDataset('train',      config['seq_len'], config.get('num_train'))
    val_ds   = TinyStoriesDataset('validation', config['seq_len'], config.get('num_val'))
    train_loader = DataLoader(train_ds, batch_size=config['batch_size'],
                              shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=config['batch_size'],
                              shuffle=False, num_workers=2, pin_memory=True)

    # ── Optimiser setup ───────────────────────────────────────────────────
    # Muon: 2-D weight matrices (excludes embedding)
    # AdamW: embeddings + LayerNorm (1-D) + centroids (treat as 2-D → Muon)
    emb_ids      = {id(p) for p in model.embedding.parameters()}
    muon_params  = [p for p in model.parameters() if p.ndim >= 2 and id(p) not in emb_ids]
    adamw_params = [p for p in model.parameters() if p.ndim < 2  or  id(p) in emb_ids]

    optimizer       = torch.optim.Muon(muon_params,  lr=config['lr'])
    optimizer_adamw = torch.optim.AdamW(
        adamw_params, lr=config['lr'],
        weight_decay=config['weight_decay'], betas=(0.9, 0.95)
    )

    steps_per_epoch = len(train_loader) * (config['seq_len'] // config['chunk_size'])
    total_steps     = config['epochs'] * steps_per_epoch
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=total_steps, eta_min=config['lr'] * 0.1
    )
    scheduler_adamw = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_adamw, T_max=total_steps, eta_min=config['lr'] * 0.1
    )
    scaler = GradScaler('cuda')

    global_step = 0

    for epoch in range(config['epochs']):
        model.train()
        epoch_lm = epoch_ks = epoch_norm = epoch_rep = n_chunks = 0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{config["epochs"]}')

        for input_ids, target_ids in pbar:
            input_ids  = input_ids.to(dev)
            target_ids = target_ids.to(dev)
            batch_size = input_ids.shape[0]
            seq_len    = input_ids.shape[1]

            h = model.init_state(batch_size, dev)

            for chunk_start in range(0, seq_len, config['chunk_size']):
                chunk_end = min(chunk_start + config['chunk_size'], seq_len)

                optimizer.zero_grad()
                optimizer_adamw.zero_grad()

                chunk_lm   = torch.zeros(1, device=dev)
                chunk_norm = torch.zeros(1, device=dev)
                all_gate_scores = []
                # Expert utilisation counter (int, no grad)
                expert_counts = [0] * n_experts
                # For routing distance logging
                dist_sum   = 0.0
                weight_ent = 0.0
                n_tok = 0

                with autocast('cuda'):
                    for t in range(chunk_start, chunk_end):
                        all_logits, gate_scores, h_new, routing_infos = model.forward_token(
                            input_ids[:, t], h, config['max_depth']
                        )
                        lm_t   = compute_lm_loss(
                            all_logits, gate_scores, target_ids[:, t], config['max_depth']
                        )
                        norm_t = norm_regularization_loss(h_new)

                        chunk_lm   = chunk_lm   + lm_t
                        chunk_norm = chunk_norm + norm_t
                        all_gate_scores.extend(gate_scores)

                        # Accumulate routing diagnostics (detached)
                        for rinfo in routing_infos:
                            # Expert utilisation
                            for e_idx in rinfo['topk_idx'].view(-1).tolist():
                                expert_counts[e_idx] += 1
                            # Mean routing distance (to nearest centroid)
                            dist_sum   += rinfo['distances'].min(dim=-1).values.mean().item()
                            # Weight entropy (concentration measure)
                            w     = rinfo['weights'].clamp(min=1e-8)
                            ent   = -(w * w.log()).sum(dim=-1).mean().item()
                            weight_ent += ent

                        h     = h_new
                        n_tok += 1

                    chunk_lm   = chunk_lm   / n_tok
                    chunk_norm = chunk_norm / n_tok

                    # Centroid repulsion (differentiable, acts on centroids directly)
                    centroids = model.processing_layer.router.centroids
                    rep_loss  = centroid_repulsion_loss(centroids, margin=config['repulsion_margin'])

                    ks_loss, ks_std_loss = ks_anchoring_loss(all_gate_scores)
                    loss = (
                        chunk_lm
                        + config['lambda_ks']   * ks_loss
                        + config['lambda_norm'] * chunk_norm
                        + config['lambda_rep']  * rep_loss
                    )

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                scaler.unscale_(optimizer_adamw)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config['max_grad_norm'])
                scaler.step(optimizer)
                scaler.step(optimizer_adamw)
                scaler.update()
                scheduler.step()
                scheduler_adamw.step()

                h = h.detach()

                epoch_lm   += chunk_lm.item()
                epoch_ks   += ks_loss.item()
                epoch_norm += chunk_norm.item()
                epoch_rep  += rep_loss.item()
                n_chunks   += 1
                global_step += 1

                if experiment and global_step % config['log_every'] == 0:
                    # Gate distribution
                    if all_gate_scores:
                        sc = torch.cat([s.detach().flatten() for s in all_gate_scores])
                        experiment.log_metric('gate_mean', sc.mean().item(), step=global_step)
                        experiment.log_metric('gate_std',  sc.std().item(),  step=global_step)

                    # SSM state norm
                    h_norm = torch.norm(h, dim=-1).mean().item()
                    experiment.log_metric('h_norm_mean', h_norm, step=global_step)

                    # Losses
                    experiment.log_metric('loss_lm',    chunk_lm.item(),  step=global_step)
                    experiment.log_metric('loss_ks',    ks_loss.item(),   step=global_step)
                    experiment.log_metric('loss_norm',  chunk_norm.item(),step=global_step)
                    experiment.log_metric('loss_rep',   rep_loss.item(),  step=global_step)
                    experiment.log_metric('loss_total', loss.item(),      step=global_step)
                    experiment.log_metric('lr', scheduler.get_last_lr()[0], step=global_step)

                    # Routing diagnostics
                    n_routing = n_tok * config['max_depth']
                    experiment.log_metric('routing_dist_mean',
                                          dist_sum / max(n_routing, 1), step=global_step)
                    experiment.log_metric('routing_weight_entropy',
                                          weight_ent / max(n_routing, 1), step=global_step)

                    # Per-expert utilisation (fraction of top-K slots)
                    total_dispatches = sum(expert_counts)
                    for e_idx, cnt in enumerate(expert_counts):
                        lbl = model.processing_layer.expert_labels[e_idx]
                        experiment.log_metric(
                            f'expert_{e_idx}_{lbl}_util',
                            cnt / max(total_dispatches, 1),
                            step=global_step
                        )

                    # Centroid spread: mean pairwise centroid distance
                    with torch.no_grad():
                        c    = model.processing_layer.router.centroids
                        diff = c.unsqueeze(0) - c.unsqueeze(1)
                        centroid_spread = diff.norm(dim=-1).mean().item()
                    experiment.log_metric('centroid_spread', centroid_spread, step=global_step)

            avg_lm  = epoch_lm  / max(n_chunks, 1)
            avg_ppl = math.exp(min(avg_lm, 20))
            pbar.set_postfix({
                'lm' : f'{avg_lm:.3f}',
                'ppl': f'{avg_ppl:.1f}',
                'ks' : f'{epoch_ks/max(n_chunks,1):.4f}',
                'rep': f'{epoch_rep/max(n_chunks,1):.4f}',
            })

        # ── Validation ────────────────────────────────────────────────────
        model.eval()
        val_lm_sum = 0.0
        val_n      = 0
        with torch.no_grad():
            for input_ids, target_ids in tqdm(val_loader, desc='Validation', leave=False):
                input_ids  = input_ids.to(dev)
                target_ids = target_ids.to(dev)
                h = model.init_state(input_ids.shape[0], dev)
                seq_lm = 0.0
                for t in range(input_ids.shape[1]):
                    al, gs, h, _ = model.forward_token(
                        input_ids[:, t], h, config['max_depth']
                    )
                    seq_lm += compute_lm_loss(al, gs, target_ids[:, t],
                                              config['max_depth']).item()
                val_lm_sum += seq_lm / input_ids.shape[1]
                val_n      += 1

        val_lm  = val_lm_sum / max(val_n, 1)
        val_ppl = math.exp(min(val_lm, 20))
        print(f'\nEpoch {epoch+1} | val_loss={val_lm:.4f} | val_ppl={val_ppl:.2f}')
        if experiment:
            experiment.log_metric('val_loss_lm', val_lm,  epoch=epoch)
            experiment.log_metric('val_ppl',     val_ppl, epoch=epoch)

        # ── Checkpoint ────────────────────────────────────────────────────
        ckpt = f'adaptive_ssm_moe_epoch{epoch+1}.pt'
        torch.save({
            'epoch'                     : epoch + 1,
            'model_state_dict'          : model.state_dict(),
            'optimizer_state_dict'      : optimizer.state_dict(),
            'optimizer_adamw_state_dict': optimizer_adamw.state_dict(),
            'config'                    : config,
            'val_loss'                  : val_lm,
        }, ckpt)
        print(f'Checkpoint saved: {ckpt}')
        if experiment:
            log_model(experiment, model, f'AdaptiveSSMMoE-epoch{epoch+1}')

    return model


---
## 9. Hyperparameters and Run

**Key changes vs Phase 1:**
- `d_expert = 256` — hidden dim inside each expert MLP (vs `d_mlp=512` for single Phase 1 MLP)
- `n_experts = 6` — 2× each of [GELU, SiLU, Polynomial]
- `top_k = 2` — 2 experts activated per routing step
- `router_temperature = 1.0` — RBF kernel width; lower → sharper routing, higher → softer
- `lambda_rep = 0.01` — centroid repulsion weight
- `repulsion_margin = 2.0` — minimum centroid pair distance before penalty kicks in

**Parameter budget (approximate):**
- Phase 1 ProcessingLayer: `2 × 256 × 512 = 262K`
- Phase 2 MoE (6 experts, d_expert=256): `6 × (2 × 256 × 256) = 786K` + router `256²=65K`
- Phase 2 has ~3× more expert parameters — the combinatorial routing paths provide the effective depth.


In [ ]:
config = {
    # Tokenizer
    'tokenizer'    : 'gpt2',

    # Model dimensions
    'd_embed'      : 128,
    'd_state'      : 256,
    'd_internal'   : 256,

    # MoE expert pool
    'd_expert'     : 128,   # hidden dim per MLP expert
    'n_experts'    : 24,     # 8x GELU + 8x SiLU + 8x Polynomial
    'top_k'        : 2,     # experts activated per routing step

    # Router
    'router_temperature' : 1.0,   # RBF kernel width T

    # Depth curriculum — start at 1 (identical to Phase 1 starting point)
    'max_depth'    : 1,

    # Sequence & batching
    'seq_len'      : 256,
    'batch_size'   : 512,
    'chunk_size'   : 32,

    # Optimisation
    'epochs'       : 5,
    'lr'           : 3e-4,
    'weight_decay' : 0.1,
    'max_grad_norm': 1.0,

    # Loss weights
    'lambda_ks'    : 0.01,
    'lambda_norm'  : 0.001,
    'lambda_rep'   : 0.01,     # centroid repulsion
    'repulsion_margin' : 2.0,  # minimum centroid distance before penalty

    # Data subsets
    'num_train'    : 100_000,
    'num_val'      : 4_000,

    # Logging
    'log_every'    : 50,
}

print('Config:')
for k, v in config.items():
    print(f'  {k:25s} = {v}')

model = train(config, experiment=experiment)


Config:
  tokenizer                 = gpt2
  d_embed                   = 128
  d_state                   = 256
  d_internal                = 256
  d_expert                  = 128
  n_experts                 = 24
  top_k                     = 2
  router_temperature        = 1.0
  max_depth                 = 1
  seq_len                   = 256
  batch_size                = 512
  chunk_size                = 32
  epochs                    = 5
  lr                        = 0.0003
  weight_decay              = 0.1
  max_grad_norm             = 1.0
  lambda_ks                 = 0.01
  lambda_norm               = 0.001
  lambda_rep                = 0.01
  repulsion_margin          = 2.0
  num_train                 = 100000
  num_val                   = 4000
  log_every                 = 50
Model parameters: 21,809,152
Expert pool: 24 experts (['gelu', 'silu', 'poly', 'gelu', 'silu', 'poly', 'gelu', 'silu', 'poly', 'gelu', 'silu', 'poly', 'gelu', 'silu', 'poly', 'gelu', 'silu', 'poly', 'gelu', 

Epoch 1/5: 100%|██████████| 195/195 [21:31<00:00,  6.62s/it, lm=4.545, ppl=94.2, ks=0.0297, rep=0.1509]



Epoch 1 | val_loss=2.6267 | val_ppl=13.83
Checkpoint saved: adaptive_ssm_moe_epoch1.pt


Epoch 2/5:  71%|███████▏  | 139/195 [15:21<06:09,  6.60s/it, lm=2.420, ppl=11.2, ks=0.0083, rep=0.0063]

---
## 10. Sample Generation

The Smirnov depth control (`μ`) works identically to Phase 1 — shifting the gate score
distribution to bias toward deeper or shallower processing paths through the expert mixture.


In [ ]:
checkpoint = torch.load('adaptive_ssm_moe_epoch5.pt', map_location=device)
config     = checkpoint['config']

model = AdaptiveSSMMoE(
    vocab_size  = AutoTokenizer.from_pretrained(config['tokenizer']).vocab_size,
    d_embed     = config['d_embed'],
    d_state     = config['d_state'],
    d_internal  = config['d_internal'],
    d_expert    = config['d_expert'],
    n_experts   = config['n_experts'],
    top_k       = config['top_k'],
    max_depth   = config['max_depth'],
    temperature = config['router_temperature'],
).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Loaded checkpoint (epoch {checkpoint['epoch']}, val_loss={checkpoint['val_loss']:.4f})")


In [ ]:
def generate_sample(model, prompt: str, tokenizer, max_new_tokens=150,
                    mu=0.0, temperature=0.9, top_k=50):
    dev    = next(model.parameters()).device
    tokens = tokenizer.encode(prompt, return_tensors='pt')
    output = model.generate(tokens, max_new_tokens=max_new_tokens,
                            mu=mu, temperature=temperature, top_k=top_k)
    return tokenizer.decode(output[0], skip_special_tokens=True)


tokenizer = AutoTokenizer.from_pretrained(config['tokenizer'])
tokenizer.pad_token = tokenizer.eos_token

prompt = 'Once upon a time, a little girl found a golden key in the forest.'

print('=' * 60)
print(f'Prompt: {prompt}')
print('=' * 60)

for mu, label in [(-1.0, 'Fast   (mu=-1)'), (0.0, 'Default (mu=0)'), (1.0, 'Deep   (mu=+1)')]:
    print(f'\n--- {label} ---')
    print(generate_sample(model, prompt, tokenizer, mu=mu))
